# TRACE: how to use it

1. Load a data file and turn it into TRACE's input format.
2. Fit TRACE.
3. Read the output: gate probabilities, the selected groups, predicted trajectories.
4. Save and reload a fitted model.

`data/example.csv` is a small **synthetic** dataset (random numbers) whose true answer is known, so you can check the output. Everything runs on a CPU in under a minute.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from trace_fs import TRACE

torch.set_num_threads(4)

## 1. Load the data

One row per subject. The baseline covariates come first, followed by the trajectory, one column per visit. An empty cell means that visit was not observed.

In this file, 6 feature groups (`A`–`F`) have 3 columns each, and the trajectory has 4 visits (`y_1`–`y_4`).
Only group `A` (which drives the early visits, linearly) and group `B` (which drives the late visits, quadratically) generate the trajectory. Groups `C`–`F` are noise.

In [2]:
df = pd.read_csv("data/example.csv")
df.head()

,subject_id,A_1,A_2,A_3,B_1,B_2,B_3,C_1,C_2,C_3,...,E_1,E_2,E_3,F_1,F_2,F_3,y_1,y_2,y_3,y_4
0,1,0.1257,-0.1321,0.6404,0.1049,-0.5357,0.3616,1.3040,0.9471,-0.7037,...,-2.3250,-0.2188,-1.2459,-0.7323,-0.5443,-0.3163,0.2875,-0.6441,-0.1646,-0.4663
1,2,0.4116,1.0425,-0.1285,1.3665,-0.6652,0.3515,0.9035,0.0940,-0.7435,...,-1.0096,-0.2092,-0.1592,0.5408,0.2147,0.3554,NaN,2.4053,1.4879,NaN
2,3,-0.6538,-0.1296,0.7840,1.4934,-1.2591,1.5139,1.3459,0.7813,0.2645,...,1.8016,1.3151,0.3574,-1.2083,-0.0045,0.6565,0.1605,0.1785,1.3834,2.3488
3,4,-1.2884,0.3951,0.4299,0.6960,-1.1841,-0.6617,-0.4364,-1.1698,1.7394,...,1.5835,1.3204,0.6334,-2.2035,0.0520,0.6837,-0.7259,-1.2605,0.4672,0.7907
4,5,1.0040,-0.6179,1.8220,-1.3204,-0.6615,0.9350,0.0491,2.0024,0.1885,...,-1.2777,0.6304,0.5812,1.2946,-0.7546,1.6891,NaN,1.8133,1.2400,1.0654


## 2. Input format

| input | shape | meaning |
|---|---|---|
| `X` | `(n, p)` float | baseline covariates, one row per subject |
| `Y` | `(n, T)` float | trajectory; `NaN` = visit not observed |
| `groups` | list of `G` lists | column indices of `X` that form each feature group |
| `R` *(optional)* | `(n, T)` 0/1 | observation mask; default `~isnan(Y)` |
| `times` *(optional)* | `(n, T)` float | visit times; default `1..T` |

Here the groups are read off the column-name prefix.

In [3]:
x_cols = [c for c in df.columns if c[0] in "ABCDEF"]
y_cols = ["y_1", "y_2", "y_3", "y_4"]
group_names = sorted({c.split("_")[0] for c in x_cols})

X = df[x_cols].to_numpy(float)
Y = df[y_cols].to_numpy(float)
groups = [[j for j, c in enumerate(x_cols) if c.startswith(name + "_")] for name in group_names]

print("X:", X.shape, " Y:", Y.shape, f" missing visits: {np.isnan(Y).mean():.0%}")
print("groups:", dict(zip(group_names, groups)))

X: (400, 18)  Y: (400, 4)  missing visits: 10%
groups: {'A': [0, 1, 2], 'B': [3, 4, 5], 'C': [6, 7, 8], 'D': [9, 10, 11], 'E': [12, 13, 14], 'F': [15, 16, 17]}


## 3. Fit TRACE

* `budget=2` keeps the two best groups (`B` in the paper). Use `budget=None` to keep every group whose gate probability is above 0.5.
* `use_aipw=True` replaces the trajectory with the cross-fitted AIPW pseudo-outcome, so that missing visits are handled (Sec. IV-D of the paper).

All parameters are listed in `README.md`.

In [4]:
train, test = np.arange(300), np.arange(300, 400)
model = TRACE(groups, budget=2, use_aipw=True, seed=0).fit(X[train], Y[train])

## 4. Output format

| attribute / method | shape | meaning |
|---|---|---|
| `model.group_prob_` | `(G,)` | gate probability `P(s_g > 0)` of each group |
| `model.support_` | list of `B` ints | selected groups (indices into `groups`) |
| `model.predict(X_new)` | `(m, T)` | predicted trajectories on the original scale; uses only the selected groups |

In [5]:
pd.DataFrame({"gate probability": model.group_prob_.round(3),
              "selected": [g in model.support_ for g in range(len(groups))],
              "truly active": [name in ("A", "B") for name in group_names]},
             index=group_names)

,gate probability,selected,truly active
A,0.986,True,True
B,0.996,True,True
C,0.030,False,False
D,0.032,False,False
E,0.033,False,False
F,0.031,False,False


In [6]:
Y_hat = model.predict(X[test])
print("predicted trajectories:", Y_hat.shape)
pd.DataFrame(Y_hat[:5].round(2), columns=[f"pred {c}" for c in y_cols])

predicted trajectories: (100, 4)


,pred y_1,pred y_2,pred y_3,pred y_4
0,-2.01,-0.99,-0.64,0.32
1,-0.02,2.11,4.21,6.26
2,-0.66,-0.36,-0.16,0.30
3,0.48,0.75,1.33,1.12
4,-1.73,-0.78,0.12,0.34


Check the result on held-out subjects. The noise has standard deviation 0.5, so no model can reach an MAE below about 0.40.

In [7]:
ok = ~np.isnan(Y[test])
print(f"TRACE (2 of 6 groups)  MAE = {np.abs(Y_hat - Y[test])[ok].mean():.3f}")
print(f"predict the mean       MAE = {np.abs(np.nanmean(Y[train], 0) - Y[test])[ok].mean():.3f}")
print(f"noise floor            MAE = {0.5 * np.sqrt(2 / np.pi):.3f}")

TRACE (2 of 6 groups)  MAE = 0.449
predict the mean       MAE = 1.101
noise floor            MAE = 0.399


## 5. Save and reload a fitted model

In [8]:
import joblib

joblib.dump(model, "trace_model.joblib")
reloaded = joblib.load("trace_model.joblib")
print("selected groups:", [group_names[g] for g in reloaded.support_])
print("same predictions:", np.allclose(reloaded.predict(X[test]), Y_hat))

selected groups: ['A', 'B']
same predictions: True


## 6. Other options

* **Missing visits without AIPW** (`use_aipw=False`): each missing visit is filled with that visit's training mean.
* **Groups of very different sizes**: raise `kappa_w`, which scales each group's sparsity cost by `(d_g / median d)^kappa_w`; `kappa_w=1` makes the cost linear in group size.
* **Very short trajectories** (`T` = 3–4): `teacher="pca"` or `teacher="dct"` replaces the attention teacher with a fixed rank-`k` basis, which has lower variance.
* **More stable selection**: `n_restarts=5` averages the gate probabilities over 5 restarts before keeping the top `budget` groups.

In [9]:
fast = TRACE(groups, budget=2, use_aipw=True, teacher="pca", n_restarts=5, seed=0).fit(X[train], Y[train])
print("PCA teacher, 5 restarts -> selected groups:", [group_names[g] for g in fast.support_])

PCA teacher, 5 restarts -> selected groups: ['A', 'B']
